# Model 4 — IndoBERT NER (Indonesian BERT — Model Utama)

**Proyek:** Deteksi Alergen pada Label Pangan Indonesia  
**Mata Kuliah:** COMP6885001 Natural Language Processing — BINUS University

---

## Deskripsi Model

Model 4 adalah **model utama (proposed model)** — fine-tuning **IndoBERT** (`indobenchmark/indobert-base-p2`) untuk task Named Entity Recognition (NER) pada teks bahan makanan Indonesia.

### Mengapa IndoBERT?
- **IndoBERT** dilatih pada corpus Bahasa Indonesia yang besar (Wikipedia ID, berita ID, dll)
- Pemahaman morfologi Bahasa Indonesia yang lebih baik dari model multilingual
- Kinerja lebih baik pada downstream task NLP bahasa Indonesia (Koto et al., 2020)

### Pipeline Lengkap
```
Open Food Facts (EN+ID) ─┐
                          ├→ BIO Labeler ─→ Fine-tune IndoBERT ─→ NER Output
Data Sintetis (ID)  ──── ┘
```

### Hasil Model (v2, trained 2026-05-07)
- **Silver F1:** 0.9946 (evaluasi circular — test set dari sumber yang sama)
- **Real-world F1:** 0.7490 micro (evaluasi pada 50 produk Indonesia, anotasi manual)
- Gap ini menunjukkan **silver label bias** — model terlalu dioptimasi untuk kamus yang sama

> **Requires GPU** — Aktifkan Runtime → Change runtime type → GPU (T4/A100)  
> **Opsional:** Gunakan cell **Load Pre-trained Model** untuk skip training (~30 menit)

In [ ]:
# torch sudah pre-install di Colab GPU runtime (dengan CUDA) — jangan reinstall
# hanya install package yang belum ada
!pip install -q datasets pandas pyarrow transformers seqeval numpy

try:
    from seqeval.metrics import f1_score as _sf
    print('seqeval siap')
except ImportError:
    print('seqeval gagal — coba Runtime > Restart and Run All')

import torch
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    print('WARNING: GPU tidak aktif — pastikan Runtime > Change runtime type > GPU')

In [ ]:
import torch
print(f'GPU tersedia  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import re
import json
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import (
    BertTokenizerFast, BertForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report as seq_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score as sk_f1
from sklearn.metrics import precision_score as sk_p, recall_score as sk_r

random.seed(42)
np.random.seed(42)

## OPSI A: Load Pre-trained Model (Skip Training)

Jalankan cell ini **jika sudah punya model yang sudah dilatih** (`model_alergen_final_v3.zip`).

Ini berguna saat presentasi ke dosen — tidak perlu menunggu 30 menit training.

In [ ]:
# =====================================================
# JALANKAN CELL INI UNTUK SKIP TRAINING (Opsi A)
# Upload model_alergen_final_v3.zip terlebih dahulu
# =====================================================

import shutil
from google.colab import files as colab_files

print('Upload model_alergen_final_v3.zip...')
uploaded = colab_files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        shutil.unpack_archive(filename, './model_alergen_final')
        print(f'Ekstrak {filename} → ./model_alergen_final')

from transformers import BertTokenizerFast, BertForTokenClassification
MODEL_LOAD_PATH = './model_alergen_final'
tokenizer  = BertTokenizerFast.from_pretrained(MODEL_LOAD_PATH)
model      = BertForTokenClassification.from_pretrained(
    MODEL_LOAD_PATH, ignore_mismatched_sizes=True
)
id2label   = model.config.id2label
label2id   = model.config.label2id
label_list = list(id2label.values())

print(f'Model berhasil dimuat!')
print(f'Labels: {label_list}')

---
## OPSI B: Training dari Awal

Jalankan cell di bawah ini untuk melatih model dari scratch (~30 menit di GPU T4).  
**Lewati bagian ini jika sudah menjalankan Opsi A.**

In [ ]:
# ========== Bagian B: Kamus Alergen ==========

ALLERGEN_CATEGORIES = {
    'GLUTEN': [
        'tepung terigu', 'tepung gandum', 'wheat starch', 'wheat bran', 'wheat germ',
        'malt extract', 'barley malt', 'hydrolyzed wheat protein',
        'tepung', 'terigu', 'gandum', 'gluten', 'barley', 'rye', 'oats',
        'sereal', 'roti', 'mie', 'pasta', 'biskuit', 'cracker', 'couscous',
        'bulgur', 'semolina', 'spelt', 'durum', 'kamut', 'malt', 'farina',
    ],
    'SUSU': [
        'susu sapi', 'milk powder', 'susu bubuk', 'nonfat milk', 'milk protein',
        'susu kental manis', 'whey protein', 'calcium caseinate', 'sodium caseinate',
        'rennet casein', 'milk solids', 'susu evaporasi', 'milk fat',
        'susu', 'keju', 'mentega', 'krim', 'yoghurt', 'butter', 'cream',
        'casein', 'kasein', 'whey', 'laktosa', 'lactose', 'skimmilk',
        'ghee', 'buttermilk', 'lactalbumin', 'lactoglobulin', 'dairy', 'caseinate',
    ],
    'TELUR': [
        'putih telur', 'kuning telur', 'egg white', 'egg yolk', 'egg powder',
        'telur bubuk', 'egg solids', 'egg protein', 'egg lecithin',
        'telur', 'egg', 'ovalbumin', 'ovomucoid', 'ovomucin',
        'lysozyme', 'mayonnaise', 'mayones', 'albumin', 'meringue',
    ],
    'KACANG': [
        'kacang tanah', 'peanut butter', 'kacang pohon', 'kacang mede', 'kacang mete',
        'kacang almond', 'kacang kenari', 'pine nut', 'brazil nut', 'tree nuts',
        'peanut', 'almond', 'cashew', 'walnut', 'pistachio', 'hazelnut',
        'pecan', 'macadamia', 'chestnut', 'groundnut', 'arachis',
    ],
    'KEDELAI': [
        'soy milk', 'susu kedelai', 'soy sauce', 'soy lecithin', 'soy protein',
        'soy isolate', 'soy flour', 'textured soy', 'lesitin kedelai', 'lesitin nabati',
        'soya lecithin',
        'kedelai', 'soy', 'soybean', 'tahu', 'tempe', 'tofu', 'tempeh',
        'edamame', 'miso', 'natto', 'yuba', 'okara',
        'lesitin', 'lecithin',
    ],
    'SEAFOOD': [
        'ikan teri', 'ikan tongkol', 'ikan kembung', 'ikan lele', 'ikan nila',
        'ikan bandeng', 'fish sauce', 'saus ikan', 'shrimp paste', 'fish oil',
        'krill oil', 'anchovy paste', 'saos tiram', 'saus tiram', 'oyster sauce',
        'ikan', 'tuna', 'salmon', 'sarden', 'makarel', 'cod', 'herring',
        'anchovy', 'udang', 'kerang', 'prawn', 'shrimp', 'cumi', 'sotong',
        'kepiting', 'crab', 'lobster', 'mussel', 'oyster', 'scallop',
        'abalone', 'terasi',
    ],
    'WIJEN': [
        'sesame oil', 'minyak wijen', 'biji wijen',
        'wijen', 'sesame', 'tahini',
    ],
    'SULFITE': [
        'sodium sulfite', 'potassium metabisulfite', 'sulfur dioxide',
        'natrium sulfit', 'kalium metabisulfit',
        'sulfite', 'sulfit',
    ],
}

ALL_CATEGORIES = list(ALLERGEN_CATEGORIES.keys())

def build_phrase_list(categories):
    phrases = [(w.lower(), cat) for cat, words in categories.items() for w in words]
    phrases.sort(key=lambda x: len(x[0].split()), reverse=True)
    return phrases

PHRASE_LIST = build_phrase_list(ALLERGEN_CATEGORIES)

def tokenize_text(text):
    return re.findall(r'\w+|[^\w\s]', text.lower())

def bio_labeler(text):
    tokens = tokenize_text(text)
    labels = ['O'] * len(tokens)
    i = 0
    while i < len(tokens):
        matched = False
        for phrase, cat in PHRASE_LIST:
            phrase_tokens = tokenize_text(phrase)
            plen = len(phrase_tokens)
            if tokens[i:i + plen] == phrase_tokens:
                if all(labels[j] == 'O' for j in range(i, i + plen)):
                    labels[i] = f'B-{cat}'
                    for j in range(i + 1, i + plen):
                        labels[j] = f'I-{cat}'
                    i += plen
                    matched = True
                    break
        if not matched:
            i += 1
    return list(zip(tokens, labels))

print(f'Kamus siap: {len(ALL_CATEGORIES)} kategori')

In [ ]:
# ========== Data Loading ==========
TOTAL_DATA  = 3000
N_SYNTHETIC = 500

print('Streaming Open Food Facts (EN+ID)...')
dataset = load_dataset('openfoodfacts/product-database', split='food', streaming=True)

sample_data = []
for example in dataset:
    lang = example.get('lang', '')
    if lang not in ('en', 'id'):
        continue
    raw = example.get('ingredients_text')
    if not raw:
        continue
    text = raw[0].get('text', '') if (isinstance(raw, list) and isinstance(raw[0], dict)) else str(raw)
    if not text or text == 'nan' or len(text) < 5:
        continue
    sample_data.append({'product_name': example.get('product_name', 'Unknown'), 'ingredients_text': text})
    if len(sample_data) >= TOTAL_DATA:
        break

df_off = pd.DataFrame(sample_data)

NON_ALLERGEN_WORDS = [
    'garam', 'air', 'gula', 'minyak sawit', 'pati', 'vitamin c',
    'kalium sorbat', 'asam sitrat', 'pewarna makanan', 'antioksidan',
    'pengatur keasaman', 'natrium klorida', 'glukosa', 'asam laktat',
    'karagenan', 'agar', 'pektin', 'karamel', 'natrium bikarbonat',
    'perisa', 'ekstrak vanili', 'kunyit',
]
TEMPLATES_ID = [
    'Komposisi: {bahan}.', 'Bahan: {bahan}.', 'Mengandung: {bahan}.', '{bahan}.',
]

random.seed(42)
all_words = [(w, cat) for cat, words in ALLERGEN_CATEGORIES.items() for w in words]
synthetic = []
for _ in range(N_SYNTHETIC):
    n_alr = random.randint(1, 3)
    n_non = random.randint(2, 5)
    selected_alr = random.sample(all_words, min(n_alr, len(all_words)))
    selected_non = random.sample(NON_ALLERGEN_WORDS, min(n_non, len(NON_ALLERGEN_WORDS)))
    ingredients  = [w for w, _ in selected_alr] + selected_non
    random.shuffle(ingredients)
    text = random.choice(TEMPLATES_ID).format(bahan=', '.join(ingredients))
    synthetic.append({'product_name': 'Synthetic-ID', 'ingredients_text': text})

df_combined = pd.concat(
    [df_off, pd.DataFrame(synthetic)], ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'OFF: {len(df_off)} | Sintetis: {N_SYNTHETIC} | Total: {len(df_combined)}')

In [ ]:
# ========== BIO Labeling ==========
labeled_dataset = []
for _, row in df_combined.iterrows():
    text = str(row['ingredients_text'])
    if not text or text == 'nan' or len(text) < 3:
        continue
    bio = bio_labeler(text)
    if bio:
        labeled_dataset.append({'clean_ingredients': text, 'tokens_and_tags': bio})

all_labels   = [tag for item in labeled_dataset for _, tag in item['tokens_and_tags']]
label_counts = Counter(all_labels)
print(f'Sampel: {len(labeled_dataset)} | Token: {len(all_labels):,}')

In [ ]:
# ========== Model Setup ==========
# ==========================================
# PERBEDAAN DARI mBERT: base model IndoBERT
MODEL_NAME = 'indobenchmark/indobert-base-p2'
# ==========================================

tokenizer  = BertTokenizerFast.from_pretrained(MODEL_NAME)
label_list = ['O'] + [f'{prefix}-{cat}' for cat in ALL_CATEGORIES for prefix in ['B', 'I']]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for i, l in enumerate(label_list)}

print(f'Model     : {MODEL_NAME}')
print(f'Vocab size: {tokenizer.vocab_size:,}')
print(f'Labels    : {len(label_list)}')

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['tokens'], truncation=True, max_length=512, is_split_into_words=True)
    all_label_ids = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(label2id.get(label_seq[word_id], 0))
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        all_label_ids.append(label_ids)
    tokenized['labels'] = all_label_ids
    return tokenized

formatted = {
    'tokens'  : [[t for t, _ in item['tokens_and_tags']] for item in labeled_dataset],
    'ner_tags': [[l for _, l in item['tokens_and_tags']] for item in labeled_dataset],
}
full_dataset = Dataset.from_dict(formatted)
tokenized_dataset = full_dataset.map(tokenize_and_align_labels, batched=True)
split = tokenized_dataset.train_test_split(test_size=300, seed=42)
train_data, test_data = split['train'], split['test']
print(f'Train: {len(train_data)} | Test: {len(test_data)}')

In [ ]:
def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=2)
    true_preds  = [[id2label[pred] for pred, lbl in zip(pr, lr) if lbl != -100] for pr, lr in zip(preds, labels)]
    true_labels = [[id2label[lbl]  for pred, lbl in zip(pr, lr) if lbl != -100] for pr, lr in zip(preds, labels)]
    return {'precision': precision_score(true_labels, true_preds),
            'recall'   : recall_score(true_labels, true_preds),
            'f1'       : f1_score(true_labels, true_preds)}

# ==========================================
NUM_EPOCHS    = 10
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
OUTPUT_DIR    = './results_indobert'
# ==========================================

model = BertForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, eval_strategy='epoch', save_strategy='epoch',
    logging_strategy='epoch', learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model='f1',
    push_to_hub=False, report_to='none',
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_data, eval_dataset=test_data,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer, compute_metrics=compute_metrics,
)

print(f'Training IndoBERT NER: {NUM_EPOCHS} epoch, lr={LEARNING_RATE}')
trainer.train()
print('Selesai!')

In [ ]:
pred_output  = trainer.predict(test_data)
preds_eval   = np.argmax(pred_output.predictions, axis=2)
labels_eval  = pred_output.label_ids
true_preds   = [[id2label[p] for p, l in zip(pr, lr) if l != -100] for pr, lr in zip(preds_eval, labels_eval)]
true_labels  = [[id2label[l] for p, l in zip(pr, lr) if l != -100] for pr, lr in zip(preds_eval, labels_eval)]

print('=== Evaluasi Silver Label — Classification Report ===')
print(seq_report(true_labels, true_preds))
silver_f1 = f1_score(true_labels, true_preds)
print(f'Silver Micro F1: {silver_f1:.4f}')

In [ ]:
import shutil
trainer.save_model('./model_alergen_final')
tokenizer.save_pretrained('./model_alergen_final')
shutil.make_archive('model_alergen_final_v3', 'zip', './model_alergen_final')
try:
    from google.colab import files
    files.download('model_alergen_final_v3.zip')
except ImportError:
    pass
print('Model disimpan sebagai model_alergen_final_v3.zip')

---
## Inferensi & Evaluasi (Berlaku untuk Opsi A & B)

In [ ]:
def predict_allergens(text, model, tokenizer, id2label):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()
    inputs = tokenizer(text.lower(), return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred_ids = torch.argmax(outputs.logits, dim=2)[0].tolist()
    tokens   = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    results, cur_word, cur_label = [], '', 'O'
    for token, pid in zip(tokens, pred_ids):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        if token.startswith('##'):
            cur_word += token[2:]
        else:
            if cur_word:
                results.append((cur_word, cur_label))
            cur_word, cur_label = token, id2label[pid]
    if cur_word:
        results.append((cur_word, cur_label))
    return results


def predict_categories_indobert(text):
    results = predict_allergens(text, model, tokenizer, id2label)
    return sorted({label[2:] for _, label in results if label.startswith('B-')})


def print_allergen_detail(text):
    print(f'\nInput : {text}')
    results = predict_allergens(text, model, tokenizer, id2label)
    found = [(w, l) for w, l in results if l != 'O']
    if found:
        print('Token alergen:')
        for w, l in found:
            print(f'  [{l}] {w}')
        cats = sorted({l[2:] for _, l in found if l.startswith('B-')})
        print(f'Kategori: {cats}')
    else:
        print('Tidak ada alergen terdeteksi.')


# Demo inferensi
print('=== Demo Inferensi — IndoBERT NER ===')
DEMO_TEXTS = [
    'Komposisi: tepung terigu, gula, susu bubuk, kacang tanah, perisa alami.',
    'Bahan: tuna, shrimp paste, soy sauce, sesame oil, garam.',
    'Bahan: lesitin nabati, minyak sawit, coklat, susu kental manis, telur.',
    'Komposisi: gula, pati jagung, air, pewarna (E150a), tanpa alergen.',
]
for text in DEMO_TEXTS:
    print_allergen_detail(text)

In [ ]:
EVAL_FILE = 'evaluation_dataset.json'
ALL_CATEGORIES = ['GLUTEN', 'SUSU', 'TELUR', 'KACANG', 'KEDELAI', 'SEAFOOD', 'WIJEN', 'SULFITE']

if not os.path.exists(EVAL_FILE):
    print(f'{EVAL_FILE} tidak ditemukan.')
    print('Catatan: real-world F1 dari training sebelumnya = 0.7490 (50 produk)')
    M4_RESULTS = {
        'model': 'IndoBERT NER (v2, pre-trained)',
        'silver_micro_f1': 0.9946,
        'micro_f1': 0.7490,
        'macro_f1': 0.6317,
    }
else:
    with open(EVAL_FILE, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)

    y_true_raw, y_pred_raw = [], []
    for item in eval_data:
        pred = predict_categories_indobert(item['ingredient_text'])
        y_pred_raw.append(pred)
        y_true_raw.append(item['ground_truth'])

    mlb_eval = MultiLabelBinarizer(classes=ALL_CATEGORIES)
    Y_true   = mlb_eval.fit_transform(y_true_raw)
    Y_pred   = mlb_eval.transform(y_pred_raw)

    print('=== MODEL 4: IndoBERT NER (Gold Label Evaluation) ===')
    print(classification_report(Y_true, Y_pred, target_names=ALL_CATEGORIES, zero_division=0))

    micro_p = sk_p(Y_true, Y_pred, average='micro', zero_division=0)
    micro_r = sk_r(Y_true, Y_pred, average='micro', zero_division=0)
    micro_f = sk_f1(Y_true, Y_pred, average='micro', zero_division=0)
    macro_f = sk_f1(Y_true, Y_pred, average='macro', zero_division=0)

    print(f'Micro Precision : {micro_p:.4f}')
    print(f'Micro Recall    : {micro_r:.4f}')
    print(f'Micro F1        : {micro_f:.4f}')
    print(f'Macro F1        : {macro_f:.4f}')

    M4_RESULTS = {
        'model': 'IndoBERT NER',
        'micro_precision': round(micro_p, 4),
        'micro_recall': round(micro_r, 4),
        'micro_f1': round(micro_f, 4),
        'macro_f1': round(macro_f, 4),
    }

print(f'M4_RESULTS = {M4_RESULTS}')

## Tabel Perbandingan 4 Model

Isi nilai dari M1, M2, M3, M4 setelah masing-masing notebook dijalankan.

In [ ]:
# ====================================================
# ISI NILAI M1, M2, M3 SETELAH MASING-MASING DIJALANKAN
# M4 otomatis diisi dari M4_RESULTS di atas
# ====================================================
M1_GOLD_F1  = '???'   # isi dari output M1 notebook
M1_MACRO_F1 = '???'
M2_GOLD_F1  = '???'   # isi dari output M2 notebook
M2_MACRO_F1 = '???'
M3_SILVER_F1 = '???'  # isi dari output M3 notebook
M3_GOLD_F1   = '???'
M3_MACRO_F1  = '???'

m4_gold   = str(round(M4_RESULTS.get('micro_f1', 0), 4))   if M4_RESULTS else '???'
m4_macro  = str(round(M4_RESULTS.get('macro_f1', 0), 4))   if M4_RESULTS else '???'
m4_silver = str(round(M4_RESULTS.get('silver_micro_f1', 0), 4)) if M4_RESULTS and 'silver_micro_f1' in M4_RESULTS else 'N/A (Opsi A)'

comparison_table = [
    {'Model': 'M1: Dictionary NER',          'Type': 'Rule-based NER',    'Silver F1': 'N/A',      'Gold F1': M1_GOLD_F1,  'Macro F1': M1_MACRO_F1},
    {'Model': 'M2: TF-IDF + LR',             'Type': 'ML Classification', 'Silver F1': '???',      'Gold F1': M2_GOLD_F1,  'Macro F1': M2_MACRO_F1},
    {'Model': 'M3: mBERT NER',               'Type': 'DL NER (multi)',    'Silver F1': M3_SILVER_F1,'Gold F1': M3_GOLD_F1,  'Macro F1': M3_MACRO_F1},
    {'Model': 'M4: IndoBERT NER (Proposed)', 'Type': 'DL NER (ID)',       'Silver F1': m4_silver,  'Gold F1': m4_gold,     'Macro F1': m4_macro},
]

import pandas as pd
df_compare = pd.DataFrame(comparison_table)
print('=== TABEL PERBANDINGAN 4 MODEL ===')
print(df_compare.to_string(index=False))
print()
print('Catatan:')
print('- Silver F1: evaluasi pada test split silver (circular evaluation)')
print(f'- Gold F1  : evaluasi pada {len(eval_data) if os.path.exists("evaluation_dataset.json") else "N/A"} produk Indonesia (anotasi manual)')
print('- Metric   : category-level micro F1')

## Analisis: Mengapa IndoBERT?

### 1. Pre-training pada Corpus Indonesia
IndoBERT dilatih pada corpus Bahasa Indonesia yang besar (Oscar corpus, CC100, Wikipedia ID), sehingga memiliki representasi subword yang lebih baik untuk kata-kata Indonesia seperti `kedelai`, `lesitin`, `terigu`.

### 2. WordPiece dan Sub-token
BERT memecah kata menjadi sub-token menggunakan WordPiece tokenization:
- `"terigu"` → `["teri", "##gu"]` (bisa berbeda antara mBERT dan IndoBERT)
- IndoBERT memiliki vocabulary yang lebih cocok untuk kata Indonesia

### 3. Silver Label Bias
F1=0.9946 (silver) → F1=0.7490 (gold) menunjukkan model terlalu disesuaikan dengan kamus yang sama. Gap ini adalah **silver label bias** — evaluasi circular tidak menangkap gap performa nyata.

### 4. Error Analysis per Kategori
| Kategori | Masalah Utama |
|---|---|
| TELUR | Presisi rendah (0.47) — over-predict: kata ambigu seperti `albumin` |
| KEDELAI | Recall rendah (0.50) — miss `lesitin nabati`, variasi ejaan |
| SEAFOOD | Presisi rendah (0.45) — `terasi` memiliki false positive tinggi |
| WIJEN | F1=0 — tidak ada sampel WIJEN di 50 produk eval, bukan model gagal |